In [62]:
import anthropic
import base64
import os
import pandas as pd
from pathlib import Path
import time
import json

In [64]:

# reading the API key from file
with open("fashionAPI.txt", "r") as f:
    api_key = f.read().strip()  # .strip() removes any extra spaces/newlines

client = anthropic.Anthropic(api_key=api_key)

# checking to see if it works
message = client.messages.create(
    model="claude-haiku-4-5-20251001",
    max_tokens=1024,
    messages=[
        {"role": "user", "content": "Say hello and confirm you're working!"}
    ]
)

print(message.content[0].text)


Hello! 👋 Yes, I'm working perfectly and ready to help! What can I do for you today?


In [66]:
#making a new file so my api key isn't published to github
%%writefile .gitignore
fashionAPI.txt
__pycache__/
*.pyc
.DS_Store
.ipynb_checkpoints/

SyntaxError: invalid syntax (183073272.py, line 4)

In [72]:

# Read your first fashion image
with open("exampleML.jpg", "rb") as image_file:
    image_data = base64.b64encode(image_file.read()).decode("utf-8")

# Analyze it
message = client.messages.create(
    model="claude-haiku-4-5-20251001",
    max_tokens=1024,
    messages=[
        {
            "role": "user",
            "content": [
                {
                    "type": "image",
                    "source": {
                        "type": "base64",
                        "media_type": "image/jpeg",
                        "data": image_data,
                    },
                },
                {
                    "type": "text",
                    "text": """Analyze this fashion image and provide:
                    1. Dominant colors (list 2-3 main colors)
                    2. Silhouette type (e.g., oversized, fitted, A-line, etc.)
                    3. Key garment types (e.g., blazer, dress, pants)
                    4. Fabric textures (e.g., silk, denim, knit)
                    5. Notable styling elements (accessories, layering, etc.)
                    6. Patterns (polka dots, stripes, solids, florals, etc.)
                    7. Overall aesthetic/vibe (e.g., minimalist, avant-garde, streetwear)
                    
                    Format your response clearly with labeled sections."""
                }
            ],
        }
    ],
)

print(message.content[0].text)

# Fashion Analysis

## 1. Dominant Colors
- **Cream/Off-white** (primary)
- **Black** (secondary)
- **Purple/Magenta** (accent)

## 2. Silhouette Type
**Oversized/Relaxed with Fitted Elements**
- Loose, oversized windbreaker jacket
- Fitted black pants with tapered leg
- Contrast between relaxed upper and structured lower

## 3. Key Garment Types
- Oversized utility windbreaker/jacket
- Black cargo/utility pants with multiple zippers
- Shirtless/bare chest styling
- Black combat-style boots

## 4. Fabric Textures
- **Nylon/Technical fabric** (jacket - matte, lightweight)
- **Leather or faux leather** (pants - sleek, structured)
- **Rubber/Studded material** (boots - textured, grip-sole)
- **Cotton/knit blend** (inner collar details)

## 5. Notable Styling Elements
- **Sunglasses** (wraparound, sporty style)
- **Chain necklace** with pendant
- **Pearl necklace** (luxury contrast)
- **Waist belt** (white/silver hardware)
- **Neon green gloves** (accent detail)
- **Layered jewelry** (mixe

Now that we know the API key can accurately analyze the image, we can find and use a dataset so we can analyze trends through seasons, years, etc

In [ ]:
#Now i compiled a dataset of calvin klein's latest ready-to-wear fall fashion line. I am going to test the API key on a dataset like that.
from PIL import Image
import io

def analyze_fashion_image(image_path):
    with Image.open(image_path) as img:
        img = img.convert("RGB")
        img.thumbnail((1568, 1568))
        buffer = io.BytesIO()
        img.save(buffer, format="JPEG", quality=85)
        image_data = base64.b64encode(buffer.getvalue()).decode("utf-8")

    message = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=1024,
        messages=[
            {
                "role": "user",
                "content": [
                    {
                        "type": "image",
                        "source": {
                            "type": "base64",
                            "media_type": "image/jpeg",
                            "data": image_data,
                        },
                    },
                    {
                        "type": "text",
                        "text": """Analyze this fashion runway look. Return your response as valid JSON with this exact structure:
                        {
                            "colors": ["color1", "color2", "color3"],
                            "silhouette": "type here",
                            "garments": ["garment1", "garment2"],
                            "fabrics": ["fabric1", "fabric2"],
                            "patterns": "pattern type or none",
                            "aesthetic": "style here"
                        }
                        
                        Return only the raw JSON with no markdown formatting, no code blocks, no backticks, and no other text."""
                    }
                ],
            }
        ],
    )
    
    return message.content[0].text
 
def parse_response(raw_text):
    """BUG FIX 2: Strip markdown fences before parsing JSON"""
    cleaned = raw_text.strip()
    # Remove ```json ... ``` or ``` ... ``` wrappers if present
    if cleaned.startswith("```"):
        cleaned = cleaned.split("```")[1]          # grab content after first ```
        if cleaned.startswith("json"):
            cleaned = cleaned[4:]                  # strip the word "json"
    cleaned = cleaned.strip()
    return json.loads(cleaned)
 

# Process each image
seasons = {
    "fw_2023": "/Users/eilymacritchie/Desktop/MQE/ML project/fw_2023",
    "ss_2024": "/Users/eilymacritchie/Desktop/MQE/ML project/ss_2024",
    "fw_2024": "/Users/eilymacritchie/Desktop/MQE/ML project/fw_2024",
    "ss_2025": "/Users/eilymacritchie/Desktop/MQE/ML project/ss_2025",
    "fw_2025": "/Users/eilymacritchie/Desktop/MQE/ML project/fw_2025",
    "ss_2026": "/Users/eilymacritchie/Desktop/MQE/ML project/ss_2026",
    "fw_2026": "/Users/eilymacritchie/Desktop/MQE/ML project/fw_2026",
    "ss_2023": "/Users/eilymacritchie/Desktop/MQE/ML project/ss_2023",
}
results = []
success_count = 0

for season, folder in seasons.items():
    image_files = list(Path(folder).glob("*.jpg")) + \
                  list(Path(folder).glob("*.jpeg")) + \
                  list(Path(folder).glob("*.png")) + \
                  list(Path(folder).glob("*.webp"))
    
    print(f"\n--- {season}: found {len(image_files)} images ---")
    
    for i, img_path in enumerate(image_files):
        print(f"Processing {i+1}/{len(image_files)}: {img_path.name}")
        
        try:
            raw_response = analyze_fashion_image(img_path)
            analysis = parse_response(raw_response)
            
            results.append({
                'filename':   img_path.name,
                'season':     season,
                'designer':   'Valentino',
                'colors':     ', '.join(analysis.get('colors', [])),
                'silhouette': analysis.get('silhouette'),
                'garments':   ', '.join(analysis.get('garments', [])),
                'fabrics':    ', '.join(analysis.get('fabrics', [])),
                'patterns':   analysis.get('patterns'),
                'aesthetic':  analysis.get('aesthetic')
            })
            
            success_count += 1
            print(f"✓ Colors: {analysis.get('colors')} | Silhouette: {analysis.get('silhouette')}")
            
        except json.JSONDecodeError as e:
            print(f"✗ JSON Parse Error: {e}")
            print(f"  Raw response was: {raw_response[:150]}")
            results.append({
                'filename': img_path.name, 'season': season, 'designer': 'Valentino',
                'colors': None, 'silhouette': None, 'garments': None,
                'fabrics': None, 'patterns': None, 'aesthetic': f"ERROR: {e}"
            })
            
        except Exception as e:
            print(f"✗ Error: {e}")
            results.append({
                'filename': img_path.name, 'season': season, 'designer': 'Valentino',
                'colors': None, 'silhouette': None, 'garments': None,
                'fabrics': None, 'patterns': None, 'aesthetic': f"ERROR: {e}"
            })
        
        time.sleep(1)

df = pd.DataFrame(results)
df.to_csv('valentino_analysis_results.csv', index=False)

print(f"\n✓ Analysis complete! Results saved to valentino_analysis_results.csv")
print(f"Successfully analyzed {success_count} images")

In [ ]:
df = pd.read_csv('valentino_analysis_results.csv')

df.head()